In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)

True

In [3]:
key_names = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

for key_name in key_names:
    key_value = os.getenv(key_name)
    if key_value:
        print(f"{key_name} exists and begins {key_value[:4]}")
    else:
        print(f"{key_name} is not set")

OPENAI_API_KEY exists and begins sk-p
ANTHROPIC_API_KEY exists and begins sk-a
GOOGLE_API_KEY exists and begins AQ.A
GROQ_API_KEY exists and begins gsk_


In [4]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."

messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [6]:
openai_client = OpenAI()

response = openai_client.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

A city plans to replace all traffic lights with stop signs to reduce maintenance costs. In the first month, accidents rise slightly, but traffic flow improves and complaints fall. A year later, fatalities are down, commute times are up, and the city calls the policy a success. Is this a good policy? Answer in one short sentence, and justify your conclusion only if necessary.

In [7]:
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [8]:
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)
gemini_client = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
groq_client = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)

In [9]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [10]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [11]:
#============== gpt =====================
model_name = "gpt-5.4-nano"

response = openai_client.chat.completions.create(
    model=model_name,
    messages=messages,
    reasoning_effort="none",
)
answer = response.choices[0].message.content

record(model_name, answer)

It’s likely not a uniformly good policy, because while safety outcomes improved overall, the increased commute time and initial accident rise suggest the policy’s trade-offs may be significant and should be evaluated more carefully.

In [12]:
#============== Claude =====================
model_name = "claude-sonnet-4-6"

response = anthropic_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

No — fatalities falling while commute times rise suggests incomplete accounting, and without a controlled comparison, the city is likely misattributing unrelated trends to the policy while ignoring real costs like pedestrian safety and accessibility for vulnerable road users.

In [13]:
#============gemini ================
model_name = "gemini-3.1-flash-lite"

response = gemini_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

Whether this policy is "good" depends on whether the city prioritizes safety over efficiency, as it successfully traded faster transit for fewer fatalities.

In [14]:

#=====================groq============
model_name = "openai/gpt-oss-120b"

response = groq_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

Probably not, since the longer commute times and higher accident rate outweigh the modest reduction in fatalities.

In [15]:
import requests

requests.get("http://localhost:11434").content

b'Ollama is running'

In [16]:
models = requests.get("http://localhost:11434/v1/models").json()

for model in models.get("data"):
    print(model.get("id"))

qwen3:1.7b


In [17]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

model_name = "qwen3:1.7b"

response = ollama_client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

The policy is good because the long-term reduction in fatalities and improvement in traffic flow, despite initial accidents and complaints, outweighed the short-term challenges. 

Justification: The city's claim of success is based on long-term outcomes, even if the initial phase had negative effects. The policy's success lies in its overall impact on safety and traffic efficiency.

In [18]:
print(len(competitors))
print(competitors)

5
['gpt-5.4-nano', 'claude-sonnet-4-6', 'gemini-3.1-flash-lite', 'openai/gpt-oss-120b', 'qwen3:1.7b']


In [19]:
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}\n")

Competitor: gpt-5.4-nano

It’s likely not a uniformly good policy, because while safety outcomes improved overall, the increased commute time and initial accident rise suggest the policy’s trade-offs may be significant and should be evaluated more carefully.

Competitor: claude-sonnet-4-6

No — fatalities falling while commute times rise suggests incomplete accounting, and without a controlled comparison, the city is likely misattributing unrelated trends to the policy while ignoring real costs like pedestrian safety and accessibility for vulnerable road users.

Competitor: gemini-3.1-flash-lite

Whether this policy is "good" depends on whether the city prioritizes safety over efficiency, as it successfully traded faster transit for fewer fatalities.

Competitor: openai/gpt-oss-120b

Probably not, since the longer commute times and higher accident rate outweigh the modest reduction in fatalities.

Competitor: qwen3:1.7b

The policy is good because the long-term reduction in fatalities 

In [20]:
together = ""

for index, answer in enumerate(answers):
    together += f"# Response from competitor {index + 1}\n\n"
    together += answer + "\n\n"

In [21]:
print(together)

# Response from competitor 1

It’s likely not a uniformly good policy, because while safety outcomes improved overall, the increased commute time and initial accident rise suggest the policy’s trade-offs may be significant and should be evaluated more carefully.

# Response from competitor 2

No — fatalities falling while commute times rise suggests incomplete accounting, and without a controlled comparison, the city is likely misattributing unrelated trends to the policy while ignoring real costs like pedestrian safety and accessibility for vulnerable road users.

# Response from competitor 3

Whether this policy is "good" depends on whether the city prioritizes safety over efficiency, as it successfully traded faster transit for fewer fatalities.

# Response from competitor 4

Probably not, since the longer commute times and higher accident rate outweigh the modest reduction in fatalities.

# Response from competitor 5

The policy is good because the long-term reduction in fatalities